# Team Season - team_info_common

This notebook inspects the data **downloaded** from the `team_info_common` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_info_common`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_info_common"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [10]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_info_common
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_info_common
Parquet files: 30


,file,size_mb
0,team_info_common__team_id=1610612737.parquet,0.017
1,team_info_common__team_id=1610612738.parquet,0.017
2,team_info_common__team_id=1610612739.parquet,0.017
3,team_info_common__team_id=1610612740.parquet,0.017
4,team_info_common__team_id=1610612741.parquet,0.017
5,team_info_common__team_id=1610612742.parquet,0.017
6,team_info_common__team_id=1610612743.parquet,0.017
7,team_info_common__team_id=1610612744.parquet,0.017
8,team_info_common__team_id=1610612745.parquet,0.017
9,team_info_common__team_id=1610612746.parquet,0.017


In [11]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_info_common__team_id=1610612737.parquet,4,30
1,team_info_common__team_id=1610612738.parquet,4,30
2,team_info_common__team_id=1610612739.parquet,4,30
3,team_info_common__team_id=1610612740.parquet,4,30
4,team_info_common__team_id=1610612741.parquet,4,30
5,team_info_common__team_id=1610612742.parquet,4,30
6,team_info_common__team_id=1610612743.parquet,4,30
7,team_info_common__team_id=1610612744.parquet,4,30
8,team_info_common__team_id=1610612745.parquet,4,30
9,team_info_common__team_id=1610612746.parquet,4,30


In [12]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (120, 30)
Total rows: 120
Total columns: 30


In [13]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,120,30,3600,1500,41.667


In [14]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
MAX_YEAR,60,50.0
MIN_YEAR,60,50.0
OPP_PTS_RANK,60,50.0
AST_PG,60,50.0
AST_RANK,60,50.0
REB_PG,60,50.0
REB_RANK,60,50.0
PTS_PG,60,50.0
PTS_RANK,60,50.0
SEASON_ID,60,50.0


In [15]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.0
"(0.01, 0.05]",0,0.0
"(0.05, 0.1]",0,0.0
"(0.1, 0.25]",0,0.0
"(0.25, 0.5]",120,100.0
"(0.5, 0.75]",0,0.0
"(0.75, 1.0]",0,0.0


In [16]:
# Merge all teams, coalesce duplicated rows by TEAM_ID, and export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to merge.")
else:
    key_col = "TEAM_ID" if "TEAM_ID" in df_all.columns else None
    if key_col is None:
        print("Required column not found: TEAM_ID")
        print(df_all.columns)
    else:
        before_rows = len(df_all)

        # Coalesce duplicates: for each TEAM_ID, take first non-null per column
        def first_non_null(s):
            s2 = s.dropna()
            return s2.iloc[0] if len(s2) else None

        df_coalesced = (
            df_all
            .groupby(key_col, as_index=False)
            .agg({col: first_non_null for col in df_all.columns})
        )

        after_rows = len(df_coalesced)
        print(f"Rows before: {before_rows}")
        print(f"Rows after: {after_rows}")
        print(f"Removed: {before_rows - after_rows}")

        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "team_info_common.parquet"
        df_coalesced.to_parquet(out_path, index=False)
        print(f"Saved to: {out_path}")


Rows before: 120
Rows after: 30
Removed: 90
Saved to: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_info_common.parquet
